# This is a sample Jupyter Notebook

Below is an example of a code cell. 
Put your cursor into the cell and press Shift+Enter to execute it and select the next one, or click 'Run Cell' button.

Press Double Shift to search everywhere for classes, files, tool windows, actions, and settings.

To learn more about Jupyter Notebooks in PyCharm, see [help](https://www.jetbrains.com/help/pycharm/ipython-notebook-support.html).
For an overview of PyCharm, go to Help -> Learn IDE features or refer to [our documentation](https://www.jetbrains.com/help/pycharm/getting-started.html).

In [87]:
#1 Imports and Warnings Suppression

In [4]:
import requests
from bs4 import BeautifulSoup
from click import prompt
from itext2kg import iText2KG_Star
from newspaper import Article
from youtube_transcript_api import YouTubeTranscriptApi
from iText2KG.itext2kg.documents_distiller import DocumentsDistiller, CV
import yaml
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.caches import BaseCache
from pydantic import BaseModel, Field
from langchain_core.callbacks.base import Callbacks
from typing import List
from iText2KG.itext2kg import iText2KG 
import networkx as nx
import matplotlib as plt
from langchain.text_splitter import RecursiveCharacterTextSplitter
import time
import warnings
from langchain_ollama import ChatOllama, OllamaEmbeddings
from neo4j_client import Neo4jClient
import traceback
import json
import os
import re
import asyncio

from typing import List, Dict, Any, Optional
from itext2kg import iText2KG_Star
from itext2kg.logging_config import setup_logging, get_logger


from langchain_core.caches import BaseCache

ChatOllama.BaseCache = BaseCache
ChatOllama.model_rebuild()

# Suppress Pydantic schema warning
warnings.filterwarnings("ignore", category=UserWarning, module="pydantic._internal._generate_schema")

ChatOpenAI.BaseCache = BaseCache
ChatOpenAI.model_rebuild()


C:\Users\gorge\AppData\Local\Programs\Python\Python312\Lib\site-packages\pydantic\_internal\_generate_schema.py:628: UserWarning: <built-in function array> is not a Python type (it may be an instance of an object), Pydantic will allow any object with no validation since we cannot even enforce that the input is an instance of the given type. To get rid of this error wrap the type with `pydantic.SkipValidation`.
  warn(
C:\Users\gorge\AppData\Local\Programs\Python\Python312\Lib\site-packages\pydantic\_internal\_generate_schema.py:628: UserWarning: <built-in function array> is not a Python type (it may be an instance of an object), Pydantic will allow any object with no validation since we cannot even enforce that the input is an instance of the given type. To get rid of this error wrap the type with `pydantic.SkipValidation`.
  warn(


In [5]:
#2 Article Scraping Function 

In [6]:
def scrape_article(url):
    print(f"Scraping article from: {url}")

    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                      "AppleWebKit/537.36 (KHTML, like Gecko) "
                      "Chrome/114.0.0.0 Safari/537.36"
    }

    try:
        response = requests.get(url, headers=headers, timeout=30)
        response.raise_for_status()

        # Method 1: BeautifulSoup approach
        soup = BeautifulSoup(response.text, 'html.parser')
        content_div = soup.find("div", class_="entry-content ap-pattern--entry-content")

        if content_div:
            article_text = content_div.get_text(separator="\n", strip=True)
            with open("output/pure_scrum_clean.txt", "w", encoding="utf-8") as f:
                f.write(article_text)
            print("Article scraped successfully with BeautifulSoup")
        else:
            print("Specific div not found, trying newspaper3k...")

        # Method 2: Newspaper3k approach (as backup)
        article = Article(url)
        article.download()
        article.parse()

        if article.text:
            with open("output/pure_scrum_clean2.txt", "w", encoding="utf-8") as f:
                f.write(article.text)
            print("Article scraped successfully with newspaper3k")
            return article.text
        else:
            print("Failed to extract article content")
            return None

    except requests.RequestException as e:
        print(f"Error fetching article: {e}")
        return None

In [7]:
#3. YouTube Transcript Retrieval Function

In [8]:
def get_youtube_transcript(video_id):
    print(f"Getting YouTube transcript for video: {video_id}")

    try:
        transcript_list = YouTubeTranscriptApi.get_transcript(video_id)
        transcript = " ".join([item['text'] for item in transcript_list])

        with open("output/youtubeVideo.txt", "w", encoding="utf-8") as f:
            f.write(transcript)

        print("YouTube transcript downloaded successfully")
        return transcript

    except Exception as e:
        print(f"Error getting YouTube transcript: {e}")
        return None

In [9]:
#4. Configuration Loading

In [10]:
def load_config(config_path="config.yml"):
    try:
        with open(config_path, "r") as f:
            config = yaml.safe_load(f)
        return config
    except FileNotFoundError:
        print("config.yml not found. Creating sample config for Ollama...")

        sample_config = {
            'llm': {
                'model_type': 'ollama',
                'model_name': 'llama3',
                'format': 'json',
                'temperature': 0.7,
                'max_tokens': 2000,
                'timeout': 60,
                'max_retries': 3
            },
            'paths': {
                'output_dir': './output'
            }
        }

        with open(config_path, "w") as f:
            yaml.dump(sample_config, f, default_flow_style=False)

        print("Sample config.yml created. Update it if needed.")
        return sample_config


In [11]:
#5. LLM and Embeddings Initialization

In [12]:
# import os
# 
# 
# def initialize_llm(config):
#     if not config:
#         return None, None
#     
#     llm_config = config.get('llm', {})
#     api_key = llm_config.get('api_key')
#     
#     if not api_key:
#         api_key = os.getenv('OPENAI_API_KEY')
#         if not api_key:
#             print("OpenAI API key not found. Please set OPENAI_API_KEY environment variable or update config.yml")
#             return None, None
#     
#     try:
#         if llm_config.get('model_type') == "openai":
#             llm = ChatOpenAI(
#                 model=llm_config.get('model_name', 'gpt-3.5-turbo'),
#                 temperature=llm_config.get('temperature', 0.7),
#                 max_tokens=llm_config.get('max_tokens', 2000),
#                 api_key=api_key,
#                 request_timeout=llm_config.get('timeout', 60),
#                 max_retries=llm_config.get('max_retries', 3)
#             )
#             embeddings = OpenAIEmbeddings(api_key=api_key)
#             print("LLM and Embeddings initialized successfully")
#             return llm, embeddings
#         else:
#             print("Only 'openai' model type is supported")
#             return None, None
#     except Exception as e:
#         print(f"Error initializing LLM: {e}")
#         return None, None
#     

In [13]:
#5.1 Ollama

In [14]:

def initialize_llm(config):
    if not config:
        return None, None

    llm_config = config.get('llm', {})

    try:
        if llm_config.get('model_type') == "ollama":
            llm = ChatOllama(
                model=llm_config.get('model_name', 'llama3'),
                format=llm_config.get('format', 'json'),
                temperature=llm_config.get('temperature', 0.7)
            )

            embeddings = OllamaEmbeddings(model=llm_config.get('model_name', 'llama3'))

            print("LLM and Embeddings initialized successfully (Ollama)")
            return llm, embeddings
        else:
            print("Only 'ollama' model type is supported in this setup")
            return None, None
    except Exception as e:
        print(f"Error initializing LLM: {e}")
        return None, None


In [15]:
#6. Pydantic Models for Triples

In [16]:
class Triple(BaseModel):
    subject: str
    predicate: str
    object: str


class TriplesOutput(BaseModel):
    triples: List[Triple]


In [17]:
#7. Knowledge Graph Extraction Function with Retry

In [18]:
def simple_kg_extraction(text, llm, limit=8, max_retries=3):
    if not llm:
        print("LLM not initialized")
        return []

    prompt_template = """
Extract key knowledge graph triples from the following text.
Only return triples in the form (subject, predicate, object).
Focus on important entities and relationships. Avoid duplicates.
Output each triple on its own line with this exact format:
- (Entity1, relationship, Entity2)

Text: {input_text}

Return up to {limit} triples.
"""

    text_to_use = text[:1500]

    for attempt in range(max_retries):
        prompt = prompt_template.format(input_text=text_to_use, limit=limit)

        try:
            response = llm.invoke(prompt)
            lines = response.content.split('\n')
            triples = []

            for line in lines:
                line = line.strip()
                if line.startswith('-') and '(' in line and ')' in line:
                    start = line.find('(')
                    end = line.find(')')
                    if start != -1 and end != -1:
                        triple_str = line[start + 1:end]
                        parts = [part.strip() for part in triple_str.split(',')]
                        if len(parts) == 3:
                            triples.append((parts[0], parts[1], parts[2]))

            return triples

        except Exception as e:
            error_msg = str(e).lower()

            if "rate limit" in error_msg or "429" in error_msg:
                wait_time = (2 ** attempt) * 10
                print(f"Rate limit hit (attempt {attempt + 1}/{max_retries}). Waiting {wait_time}s...")
                time.sleep(wait_time)

            elif "token" in error_msg:
                print(f"Token limit exceeded. Reducing text size and retrying...")
                text_to_use = text_to_use[:len(text_to_use) // 2]
                limit = max(3, limit // 2)
                time.sleep(5)

            else:
                print(f"Error in KG extraction (attempt {attempt + 1}/{max_retries}): {e}")
                time.sleep(5)

    print(f"Failed to extract triples after {max_retries} attempts")
    return []


In [19]:
#8. Process Text in Chunks to Avoid Rate Limits

In [20]:
def process_text_in_chunks(text, llm, chunk_size=1000, chunk_overlap=50, delay_between_chunks=0):
    print(f"Processing text in chunks...")

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap
    )
    chunks = splitter.split_text(text)

    print(f"Created {len(chunks)} chunks")

    all_triples = []

    for i, chunk in enumerate(chunks, 1):
        limit = max(5, len(chunk) // 150)
        print(f"\nProcessing chunk {i}/{len(chunks)} with triple limit {limit}")

        triples = simple_kg_extraction(chunk, llm, limit=limit)
        if triples:
            all_triples.extend(triples)
            print(f"Chunk {i} → {len(triples)} triples extracted")
        else:
            print(f"Chunk {i} → No triples extracted")

        if i < len(chunks) and delay_between_chunks > 0:
            print(f"Waiting {delay_between_chunks}s before next chunk...")
            time.sleep(delay_between_chunks)

    return all_triples


In [21]:
#8. Save Triples to File and also filter duplicates

In [22]:
# # def normalize_triples(triples):
# #     return [(s.strip().lower(), p.strip().lower(), o.strip().lower()) for s, p, o in triples]
# # 
# # 
# # def filter_triples(triples):
# #     filtered = []
# #     for subj, pred, obj in triples:
# #         if subj and pred and obj and subj.lower() != 'none' and obj.lower() != 'none':
# #             filtered.append((subj, pred, obj))
# #     return filtered
# # 
# # 
# # def remove_duplicates(triples):
# #     seen = set()
# #     unique_triples = []
# #     for triple in triples:
# #         if triple not in seen:
# #             unique_triples.append(triple)
# #             seen.add(triple)
# #     return unique_triples
# # 
# # 
def save_triples(triples, filename="extracted_triples.txt"):
    with open(filename, "w", encoding="utf-8") as f:
        f.write("Extracted Knowledge Graph Triples:\n")
        f.write("=" * 40 + "\n\n")
        for i, (subject, predicate, obj) in enumerate(triples, 1):
            f.write(f"{i}. ({subject}) --[{predicate}]--> ({obj})\n")

    print(f"Triples saved to {filename}")

# 
# def normalize_triples(triples):
#     normalized = []
#     for triple in triples:
#         if len(triple) == 3:
#             subject, predicate, obj = triple
#             subject = str(subject).strip().lower()
#             predicate = str(predicate).strip().lower().replace('_', ' ')
#             obj = str(obj).strip().lower()
#             normalized.append((subject, predicate, obj))
#     return normalized
# 
# 
# def filter_triples(triples):
#     filtered = []
#     for triple in triples:
#         if len(triple) == 3:
#             subject, predicate, obj = triple
#             if len(subject) > 1 and len(predicate) > 1 and len(obj) > 1:
#                 if subject != obj:
#                     filtered.append(triple)
#     return filtered
# 
# 
# def remove_duplicates(triples):
#     return list(set(triples))
# 
# def semantic_filter(triples):
#     hallucinated_terms = {"family", "romantic", "divorce"}
#     return [
#         t for t in triples
#         if not any(term in str(t).lower() for term in hallucinated_terms)
#     ]
# 
# # def save_triples(triples, filename="extracted_triples.txt"):
# #     with open(filename, 'w', encoding='utf-8') as f:
# #         for subject, predicate, obj in triples:
# #             f.write(f"{subject}\t{predicate}\t{obj}\n")


In [23]:
#11. Plots

In [24]:
import networkx as nx
import matplotlib.pyplot as plt


def plot_triples(triples):
    G = nx.DiGraph()
    for subj, pred, obj in triples:
        G.add_node(subj)
        G.add_node(obj)
        G.add_edge(subj, obj, label=pred)

    pos = nx.spring_layout(G, k=0.5, iterations=50)

    plt.figure(figsize=(12, 8))
    nx.draw(G, pos, with_labels=True, node_color='lightblue', node_size=3000, font_size=10, arrowsize=20)

    edge_labels = {(subj, obj): pred for subj, pred, obj in triples}
    nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels, font_color='red', font_size=9)

    plt.title("Knowledge Graph Visualization")
    plt.show()


In [25]:
#9. Main Execution Logic

In [26]:
# import re
# def main():
#     print("Starting Knowledge Graph Extraction Pipeline\n")
# 
#     # Scrape article
#     # url = "https://age-of-product.com/pure-scrum/"
#     # article_text = scrape_article(url)
#     # 
#     # # Get YouTube transcript
#     # video_id = "502ILHjX9EE"
#     # youtube_text = get_youtube_transcript(video_id)
# 
#     # Choose which text to process
#     # if youtube_text:
#     #     input_text = youtube_text
#     #     print("Using YouTube transcript for KG extraction")
#     # elif article_text:
#     #     input_text = article_text
#     #     print("Using article text for KG extraction")
#     # else:
#     #     print("No text available for processing")
#     #     return
# 
#     try:
#         with open("text.txt", "r", encoding="utf-8") as f:
#             input_text = f.read()
#     except Exception as e:
#         print(f"Error reading 'text.txt': {e}")
#         return
# 
#     config = load_config()
#     llm, embeddings = initialize_llm(config)
# 
#     if not llm:
#         print("Cannot proceed without LLM initialization")
#         return
# 
#     print(f"\nInput text length: {len(input_text)} characters")
#     # all_triples = process_text_in_chunks(input_text, llm,delay_between_chunks=0)
# 
#     # shall try this next automation
#     kg = iText2KG(llm,embeddings)
# 
#     doc = {
#         "text": input_text,
#         "title": "ManualText",
#         "meta": {
#             "source": "manual_input"
#         }
#     }
#     try:
#         response = llm.invoke(f"Extract triples from text:\n\n{input_text[:1000]}")
#         print("Raw LLM response:\n", response.content)
# 
#     except Exception as e:
#         print(f"Extraction failed: {e}")
#         return
# 
#     def parse_llm_response_to_triples(text):
#         triples = []
#         pattern = re.compile(r"^\s*\d+\.\s*(.+?)\s*-\s*(.+)$")
#         for line in text.splitlines():
#             match = pattern.match(line)
#             if match:
#                 subj = match.group(1).strip()
#                 rest = match.group(2).strip()
#                 parts = rest.split(' - ')
#                 if len(parts) == 2:
#                     pred = parts[0].strip()
#                     obj = parts[1].strip()
#                 else:
#                     pred = rest
#                     obj = ""
#                 triples.append((subj, pred, obj))
#         return triples
# 
#     
#     all_triples = parse_llm_response_to_triples(response.content)
# 
#     if all_triples:
#         print(f"\nTotal triples extracted: {len(all_triples)}")
#         all_triples = normalize_triples(all_triples)
#         all_triples = filter_triples(all_triples)
#         all_triples = remove_duplicates(all_triples)
#         save_triples(all_triples)
# 
#         print(Neo4jClient.__init__)
#         neo4j_client = Neo4jClient(uri="bolt://localhost:7687", user="neo4j", password="12345678")
#         neo4j_client.create_triples(all_triples)
#         neo4j_client.close()
# 
#         print("\nSample triples:")
#         for i, (subject, predicate, obj) in enumerate(all_triples[:5], 1):
#             print(f"  {i}. ({subject}) --[{predicate}]--> ({obj})")
# 
#         if len(all_triples) > 5:
#             print(f"  ... and {len(all_triples) - 5} more triples")
# 
#         plot_triples(all_triples)
#     else:
#         print("No triples were successfully extracted")




In [35]:
async def main(input_source: str = "text.txt"):
    print("Starting Knowledge Graph Extraction with iText2KG_Star\n")

    # ============ LOAD INPUT TEXTS ============
    texts = []
    try:
        if isinstance(input_source, str) and os.path.exists(input_source):
            print(f"oading from file: {input_source}")
            with open(input_source, "r", encoding="utf-8") as f:
                content = f.read().strip()
                
                if len(content) > 2000:
                    sentences = content.split('. ')
                    current_chunk = ""
                    for sentence in sentences:
                        if len(current_chunk + sentence) <= 2000:
                            current_chunk += sentence + ". "
                        else:
                            if current_chunk:
                                texts.append(current_chunk.strip())
                            current_chunk = sentence + ". "
                    if current_chunk:
                        texts.append(current_chunk.strip())
                else:
                    texts = [content]
                    
        elif isinstance(input_source, str) and len(input_source.strip()) > 10:
            print("Using direct text input")
            texts = [input_source.strip()]
            
        else:
            print(f"Invalid input source: {input_source}")
            return []

    except Exception as e:
        print(f"Error loading input: {e}")
        return []

    texts = [text for text in texts if len(text.strip()) > 20]

    if not texts:
        print("No valid texts found. Exiting.")
        return []

    print(f"Loaded {len(texts)} text segments")
    for i, text in enumerate(texts[:2]):
        print(f"   Text {i + 1} preview: {text[:100]}...")

    # ============ INITIALIZE LLM ============
    print("\nInitializing LLM and Embeddings...")
    config = load_config()
    llm, embeddings = initialize_llm(config)

    if not llm or not embeddings:
        print("Cannot proceed without LLM initialization")
        return []

    print(f"LLM initialized: {config['llm']['model_name']}")

    # ============ KNOWLEDGE GRAPH CONSTRUCTION WITH iText2KG_Star ============
    print("\nBuilding Knowledge Graph with iText2KG_Star...")
    
    all_triples = []

    try:
        itext2kg_star = iText2KG_Star(llm_model=llm, embeddings_model=embeddings)
        print("iText2KG_Star initialized successfully")

        print(f"Building knowledge graph from {len(texts)} sections...")
        
        kg_result = await itext2kg_star.build_graph(sections=texts)

        print(f"Knowledge graph construction completed")
        print(f"Result type: {type(kg_result)}")

        # ============ EXTRACT TRIPLES FROM RESULT ============
        if hasattr(kg_result, 'relationships'):
            relationships = kg_result.relationships
            print(f"Found {len(relationships)} relationships")
        elif hasattr(kg_result, 'edges'):
            relationships = kg_result.edges
            print(f"Found {len(relationships)} edges")
        elif isinstance(kg_result, dict) and 'relationships' in kg_result:
            relationships = kg_result['relationships']
            print(f"Found {len(relationships)} relationships in dict")
        elif isinstance(kg_result, list):
            relationships = kg_result
            print(f"Found {len(relationships)} items in list")
        else:
            print("Unknown result format from iText2KG_Star")
            print(f"Available attributes: {dir(kg_result) if hasattr(kg_result, '__dict__') else 'Not an object'}")
            relationships = []

        for i, rel in enumerate(relationships):
            try:
                if isinstance(rel, dict):
                    subject = rel.get('subject') or rel.get('source') or rel.get('head')
                    predicate = rel.get('predicate') or rel.get('relation') or rel.get('type')
                    obj = rel.get('object') or rel.get('target') or rel.get('tail')
                    
                elif isinstance(rel, (tuple, list)) and len(rel) >= 3:
                    subject, predicate, obj = rel[0], rel[1], rel[2]
                    
                else:
                    print(f"Unknown relationship format at index {i}: {type(rel)}")
                    continue

                if subject and predicate and obj:
                    subject = str(subject).strip()
                    predicate = str(predicate).strip()
                    obj = str(obj).strip()
                    
                    if len(subject) > 1 and len(obj) > 1 and subject != obj:
                        all_triples.append((subject, predicate, obj))
                        print(f"({subject}) --[{predicate}]--> ({obj})")
                    
            except Exception as e:
                print(f"Error processing relationship {i}: {e}")
                continue

    except Exception as e:
        print(f"Knowledge graph construction failed: {e}")
        import traceback
        traceback.print_exc()
        return []

    # ============ POST-PROCESSING & OUTPUT ============
    print(f"\nProcessing Results...")

    if all_triples:
        original_count = len(all_triples)
        all_triples = list(set(all_triples))
        print(f"After deduplication: {len(all_triples)} triples (removed {original_count - len(all_triples)})")

        save_triples(all_triples, "extracted_triples.txt")

        print("\nEXTRACTED KNOWLEDGE GRAPH:")
        print("=" * 50)
        for i, (subject, predicate, obj) in enumerate(all_triples[:10], 1):
            print(f"  {i:2d}. ({subject}) --[{predicate}]--> ({obj})")
        
        if len(all_triples) > 10:
            print(f"  ... and {len(all_triples) - 10} more triples")

        print(f"\nSuccess! Extracted {len(all_triples)} knowledge relationships")
        
        try:
            plot_triples(all_triples[:20])  
        except Exception as e:
            print(f"Visualization skipped: {e}")
        
        return all_triples

    else:
        print("\nNo triples were successfully extracted")
        return []


In [36]:
#10. Run the main() function

In [37]:
if __name__ == "__main__":
    import nest_asyncio
    nest_asyncio.apply()

    async def run_star_pipeline():
        triples = await main("text.txt")
        print(f"\nPipeline completed! Found {len(triples)} triples")
        return triples

    loop = asyncio.get_event_loop()
    triples = loop.run_until_complete(run_star_pipeline())


Starting Knowledge Graph Extraction with iText2KG_Star

oading from file: text.txt
Loaded 1 text segments
   Text 1 preview: Let's talk about Agile Software development from the perspective of the Product Owner Here's Pat, sh...

Initializing LLM and Embeddings...
LLM and Embeddings initialized successfully (Ollama)
LLM initialized: llama3

Building Knowledge Graph with iText2KG_Star...
iText2KG_Star initialized successfully
Building knowledge graph from 1 sections...
[2025-08-06 19:55:58] [    INFO] [itext2kg.itext2kg.itext2kg_star] ------- Extracting Relations and Deriving Entities from Document 1
[2025-08-06 19:55:58] [ WARNING] [itext2kg.itext2kg.irelations_extraction.simple_direct_irelations_extractor] Not Formatted in the desired format. Error occurred: registry.ollama.ai/library/llama3:latest does not support tools (status code: 400). Retrying... (Attempt 1/5)
[2025-08-06 19:55:59] [ WARNING] [itext2kg.itext2kg.irelations_extraction.simple_direct_irelations_extractor] Not Format

Traceback (most recent call last):
  File "C:\Users\gorge\AppData\Local\Temp\ipykernel_22444\805738659.py", line 71, in main
    kg_result = await itext2kg_star.build_graph(sections=texts)
                ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\gorge\AppData\Local\Programs\Python\Python312\Lib\site-packages\itext2kg\itext2kg_star.py", line 62, in build_graph
    global_entities, global_relationships = await self.simple_irelations_extractor.extract_relations_and_derive_entities(
                                            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\gorge\AppData\Local\Programs\Python\Python312\Lib\site-packages\itext2kg\irelations_extraction\simple_direct_irelations_extractor.py", line 83, in extract_relations_and_derive_entities
    raise ValueError("Failed to extract relationships after multiple attempts.")
ValueError: Failed to extract relationships after multiple attempts.
